<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 01. Variables Categóricas: Traduciendo Texto a Números
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 06
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/06%20-%20Feature%20Engineering/Para%20Dummies/01_Variables_Categoricas_Feature_Engineering_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del módulo 01 de Feature Engineering. El cuaderno principal usó el dataset de viviendas de Melbourne y comparó tres enfoques con un modelo de *Random Forest*; aquí vamos a entender **la misma idea**, pero con tablas pequeñas que puedes leer de un vistazo.

Al terminar podrás explicar, con tus propias palabras:
1. Por qué los modelos de Machine Learning no pueden recibir texto directamente.
2. Qué hace cada uno de los tres enfoques básicos: eliminar, codificación ordinal y codificación one-hot.
3. Cuándo conviene usar cada uno.
4. Qué es, en su forma más simple, un "Target Encoding" (lo veremos a fondo en el siguiente cuaderno).

---
## 1. El problema: el modelo solo entiende números 🔢

Imagina que le entregas a un cocinero una receta escrita **en un idioma que no habla**. No importa qué tan buena sea la receta: el cocinero no puede seguirla.

Así funcionan casi todos los modelos de Machine Learning: solo "hablan" números. Si una columna de tu tabla contiene texto — como `"sedan"`, `"suv"` o `"bueno"`, `"malo"` — el modelo simplemente no la entiende, y muchas veces ni siquiera te dejará entrenarlo hasta que la traduzcas.

A esas columnas de texto con un número limitado de valores posibles las llamamos **variables categóricas**. "Traducirlas" a números, sin perder su significado, es el trabajo de este cuaderno.

---
## 2. Nuestro dataset de juguete 🚗

Vamos a trabajar con una tabla pequeña e inventada de carros usados, para poder ver **cada fila con nuestros propios ojos**. Tiene dos columnas de texto (categóricas) y dos columnas numéricas:

- `tipo`: `"sedan"`, `"suv"` o `"hatchback"` — no existe un orden natural entre ellas (es **nominal**).
- `condicion`: `"malo"`, `"regular"`, `"bueno"` o `"excelente"` — aquí sí hay un orden claro (es **ordinal**).
- `kilometraje_mil`: kilómetros recorridos, en miles.
- `precio_millones`: precio de venta, en millones de pesos.

In [ ]:
import pandas as pd

vehiculos = pd.DataFrame({
    "tipo": ["sedan", "suv", "hatchback", "suv", "sedan", "hatchback", "suv"],
    "condicion": ["regular", "bueno", "malo", "excelente", "bueno", "regular", "excelente"],
    "kilometraje_mil": [50, 30, 80, 20, 45, 90, 15],
    "precio_millones": [40, 65, 28, 80, 52, 30, 85]
})

vehiculos

### 🤔 ¿Qué acaba de pasar?

- Empezamos importando `pandas` (`import pandas as pd`), la herramienta que usaremos para manipular tablas en todo este cuaderno.
- Creamos un `DataFrame` de 7 carros con dos columnas de texto (`tipo`, `condicion`) y dos numéricas.
- Si intentaras entrenar un modelo de Machine Learning directamente con esta tabla, fallaría al toparse con `tipo` y `condicion`: son texto, no números.
- A continuación probaremos tres formas distintas de resolver este problema.

---
## 3. Enfoque 1: Eliminar las columnas de texto ❌

La solución más simple es la más drástica: **botar** las columnas que no son números. Es como si, al no entender el idioma de la receta, simplemente **arrancaras esa página** y cocinaras solo con las instrucciones que sí entiendes.

Funciona, y es rapidísimo de aplicar — pero si esas columnas tenían información valiosa (¿el tipo de carro no influye en el precio?), la estamos perdiendo por completo.

In [ ]:
# Eliminamos las columnas de texto (categóricas)
vehiculos_sin_texto = vehiculos.select_dtypes(exclude=["object"])

print("Columnas originales:", list(vehiculos.columns))
print("Columnas despues de eliminar texto:", list(vehiculos_sin_texto.columns))
vehiculos_sin_texto

### 🤔 ¿Qué acaba de pasar?

- `select_dtypes(exclude=["object"])` se queda solo con las columnas cuyo tipo **no** es texto (`object`), es decir, las numéricas.
- Perdimos `tipo` y `condicion` por completo — junto con cualquier pista que dieran sobre el precio.
- Este enfoque solo es recomendable cuando de verdad crees que la columna no aporta información útil.

---
## 4. Enfoque 2: Codificación Ordinal 🔢

Piensa en las medallas olímpicas: oro, plata y bronce **sí tienen un orden** (oro > plata > bronce). Cuando una variable categórica tiene un orden natural — como `condicion`: `"malo" < "regular" < "bueno" < "excelente"` — podemos reemplazar cada categoría por un número que respete ese orden: `0, 1, 2, 3`.

A esto se le llama **codificación ordinal (Ordinal Encoding)**, y funciona muy bien con variables **ordinales** (las que sí tienen jerarquía).

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# Indicamos explicitamente el orden de las categorias, de peor a mejor
orden_condicion = [["malo", "regular", "bueno", "excelente"]]
codificador_ordinal = OrdinalEncoder(categories=orden_condicion)

vehiculos_ordinal = vehiculos.copy()
vehiculos_ordinal["condicion_codificada"] = codificador_ordinal.fit_transform(vehiculos[["condicion"]])

vehiculos_ordinal[["condicion", "condicion_codificada"]]

### 🤔 ¿Qué acaba de pasar?

- Le dijimos a `OrdinalEncoder` cuál es el orden correcto de las categorías: de `"malo"` (peor) a `"excelente"` (mejor).
- `fit_transform` reemplazó cada texto por su posición en esa lista: `"malo"` → `0`, `"regular"` → `1`, `"bueno"` → `2`, `"excelente"` → `3`.
- Ahora el modelo puede "ver" que `"excelente"` (3) está más lejos de `"malo"` (0) que `"regular"` (1) — justo la relación que existe en la vida real.
- ⚠️ Si le hubiéramos aplicado esto a `tipo` (`"sedan"`, `"suv"`, `"hatchback"`), estaríamos **inventando un orden que no existe** — ¿por qué sería un `"suv"` mayor o menor que un `"sedan"`? Para esos casos usamos el siguiente enfoque.

---
## 5. Enfoque 3: Codificación One-Hot 🎯

Para categorías **sin orden** (nominales) como `tipo`, usamos otra idea: crear una columna nueva **por cada categoría posible**, como un panel de interruptores. Si el carro es un `"suv"`, encendemos (`1`) el interruptor `"suv"` y dejamos apagados (`0`) los demás.

A esto se le llama **One-Hot Encoding**: ninguna categoría queda "más cerca" o "más lejos" de otra, porque cada una tiene su propio interruptor independiente.

In [ ]:
# pd.get_dummies crea una columna binaria (0/1) por cada categoria de "tipo"
vehiculos_onehot = pd.get_dummies(vehiculos, columns=["tipo"], prefix="tipo")

vehiculos_onehot.head()

### 🤔 ¿Qué acaba de pasar?

- `pd.get_dummies(..., columns=["tipo"])` reemplazó la columna `tipo` por tres columnas nuevas: `tipo_sedan`, `tipo_suv` y `tipo_hatchback`.
- En cada fila, solo una de esas tres columnas vale `1` (la categoría real de ese carro); las demás valen `0`.
- Esto evita inventar un orden falso, pero tiene un costo: si una columna tuviera **cientos** de categorías distintas (por ejemplo, códigos postales), terminaríamos con cientos de columnas nuevas — la mayoría llenas de ceros. Por eso One-Hot Encoding se recomienda solo para columnas con pocas categorías (por lo general, menos de 15).

---
## 6. ¿Cuál enfoque elegir? 🧭

| Enfoque | ¿Cuándo usarlo? | Riesgo si lo usas mal |
|---|---|---|
| Eliminar | La columna de texto realmente no aporta información | Perder información valiosa |
| Ordinal | La categoría tiene un orden natural (ej. `malo < regular < bueno`) | Inventar un orden falso en una variable sin orden |
| One-Hot | La categoría no tiene orden y tiene pocas opciones (< 15) | Crear demasiadas columnas si hay muchas categorías distintas |

¿Y si una columna tiene **cientos** de categorías, como una marca de carro o un código postal, y tampoco tiene un orden claro? Ahí es donde entra un cuarto enfoque, mucho más potente: reemplazar cada categoría por un número relacionado con lo que queremos predecir.

---
## 7. Un adelanto: codificar con el precio promedio 💰

En vez de inventar un orden o crear un interruptor por categoría, podemos reemplazar cada `tipo` de carro por **el precio promedio de ese tipo** en nuestros datos. A esto se le llama, de forma simple, **Target Encoding** (codificación por el objetivo): usamos información de la variable que queremos predecir (`precio_millones`) para codificar la categoría.

In [ ]:
# Precio promedio por tipo de carro
vehiculos["tipo_precio_promedio"] = vehiculos.groupby("tipo")["precio_millones"].transform("mean")

vehiculos[["tipo", "precio_millones", "tipo_precio_promedio"]]

### 🤔 ¿Qué acaba de pasar?

- `groupby("tipo")["precio_millones"].transform("mean")` calcula el precio promedio de cada `tipo` de carro y se lo asigna a **todas** las filas de ese tipo.
- Por ejemplo, todos los `"suv"` recibieron el mismo número: el precio promedio de los `"suv"` en la tabla.
- Este truco es muy poderoso, pero también **peligroso** si no se usa con cuidado: ¿qué pasa si un tipo de carro solo aparece una vez? Su "promedio" sería simplemente su propio precio — una trampa de sobreajuste. Ese problema, y cómo solucionarlo con **suavizado**, es exactamente el tema del siguiente cuaderno.

---
## 8. Resumen relámpago ⚡

| Idea | En una frase |
|---|---|
| Variable categórica | Columna de texto con un número limitado de valores posibles. |
| Eliminar | Botar la columna de texto; simple, pero se pierde información. |
| Ordinal Encoding | Reemplazar categorías **con orden** por números que respetan ese orden (0, 1, 2...). |
| One-Hot Encoding | Crear una columna "interruptor" (0/1) por cada categoría **sin orden**; ideal si hay pocas categorías. |
| Target/Mean Encoding | Reemplazar cada categoría por un número relacionado con el objetivo (ej. el promedio del precio). |

➡️ **Siguiente paso:** en el cuaderno [02 - Target Encoding y Suavizado (Para Dummies)](02_Target_Encoding_y_Suavizado_Feature_Engineering_Dummies.ipynb) descubrirás por qué el "adelanto" de la sección 7 puede ser riesgoso, y cómo arreglarlo con una técnica llamada **suavizado**.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos (Edición Para No Ingenieros)</i>
  </p>
</div>